# 08b · Review runs on **RTX PRO 6000**
**Block A**: Tversky control, UPerNet (ConvNeXt-T), 10 LOIO folds (≈ 2 h).
Specification: `specifications/03_spec_recall_control_topology.md` (written before these runs; SHA-256 in `specifications/README.md`).
**Use an RTX PRO 6000** (same GPU as the seed-0 UPerNet LOIO runs). Resumable.

In [ ]:
# --- Setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/gwfss_stem_seg'
DATA_ROOT = '/content/gwfss'                      # local disk: fast I/O, rebuilt each session
SPLIT_FILE = f'{PROJECT}/splits/splits.json'      # frozen once, shared by all runs
RESULTS = f'{PROJECT}/results'                    # checkpoints, logs, metrics (on Drive)

import sys, os
sys.path.insert(0, f'{PROJECT}/src')
!pip -q install segmentation-models-pytorch==0.5.0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from gwfss_stem import data
manifest = data.prepare_dataset(DATA_ROOT)   # downloads GWFSS_v1.0 from Hugging Face (~600 MB) if not present
print(len(manifest), 'images')

In [ ]:
from gwfss_stem import plan, engine
import pandas as pd
def status(r):
    d = os.path.join(RESULTS, engine.run_name({**engine.DEFAULTS, **r}))
    if os.path.exists(f'{d}/final.pt'): return 'done'
    if os.path.exists(f'{d}/RUNNING'): return 'running/locked'
    if os.path.exists(f'{d}/last.pt'): return 'partial'
    return 'pending'
def show(runs):
    t = pd.DataFrame([{**r, 'status': status(r)} for r in runs]); display(t.status.value_counts()); display(t)
def run_all(runs):
    for r in runs:
        cfg = {k: v for k, v in r.items() if k != 'sq'}
        run_dir = engine.train_run(cfg, DATA_ROOT, SPLIT_FILE, RESULTS)
        if run_dir:
            engine.evaluate_run(run_dir, DATA_ROOT, SPLIT_FILE, subset='test', tag='ss')
            engine.offset_sweep(run_dir, DATA_ROOT, SPLIT_FILE, subset='val')
            engine.offset_sweep(run_dir, DATA_ROOT, SPLIT_FILE, subset='test')
            engine.topology_sweep(run_dir, DATA_ROOT, SPLIT_FILE, subset='test')
    print('done')


In [ ]:
runsA = [r for r in plan.review_plan() if r['sq'] == 'R-TV' and r['backbone'] == 'upernet_convnext_t']
show(runsA)
run_all(runsA)